In [1]:
import pandas as pd
from finvizfinance.quote import finvizfinance
import yfinance as yf
from IPython.display import display
from finvizfinance.screener.overview import Overview
from finvizfinance.screener.valuation import Valuation
from finvizfinance.screener.financial import Financial
print("All instaled")

All instaled


In [2]:
stock = finvizfinance('tsla')

In [3]:
stock.ticker_fundament()

{'Company': 'Tesla Inc',
 'Sector': 'Consumer Cyclical',
 'Industry': 'Auto Manufacturers',
 'Country': 'USA',
 'Exchange': 'NASD',
 'Index': 'NDX, S&P 500',
 'P/E': '335.12',
 'EPS (ttm)': '1.08',
 'Insider Own': '28.18%',
 'Shs Outstand': '3.75B',
 'Perf Week': '-3.10%',
 'Market Cap': '1353.09B',
 'Forward P/E': '137.94',
 'EPS next Y': '2.61',
 'Insider Trans': '-0.02%',
 'Shs Float': '2.69B',
 'Perf Month': '-11.17%',
 'Enterprise Value': '1324.09B',
 'PEG': '4.55',
 'EPS next Q': '0.40',
 'Inst Own': '43.20%',
 'Short Float': '2.26%',
 'Perf Quarter': '-19.82%',
 'Income': '3.79B',
 'P/S': '14.27',
 'EPS this Y': '18.56%',
 'Inst Trans': '0.60%',
 'Short Ratio': '0.99',
 'Perf Half Y': '-21.52%',
 'Sales': '94.83B',
 'P/B': '16.47',
 'EPS next Y Percentage': '32.82%',
 'ROA': '2.92%',
 'Short Interest': '60.86M',
 'Perf YTD': '-19.82%',
 'Book/sh': '21.90',
 'P/C': '30.44',
 'EPS next 5Y': '30.30%',
 'ROE': '4.89%',
 '52W High': '498.83 -27.71%',
 'Perf Year': '27.53%',
 'Cash/sh

In [4]:
foverview = Overview()
filters_dict = {'Exchange':'AMEX','Sector':'Basic Materials'}
foverview.set_filter(filters_dict=filters_dict)
df = foverview.screener_view()
df.head()

/opt/anaconda3/envs/quant_finance/lib/python3.12/site-packages/finvizfinance/screener/base.py:134: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  return pd.concat([df, pd.DataFrame(frame)], ignore_index=True)


,Ticker,Company,Sector,Industry,Country,Market Cap,P/E,Price,Change,Volume
0,ASM,Avino Silver & Gold Mines Ltd,Basic Materials,Other Precious Metals & Mining,Canada,1.100000e+09,37.49,6.53,-0.0091,4259514.0
1,AUST,Austin Gold Corp,Basic Materials,Gold,Canada,2.109000e+07,NaN,1.54,0.0199,82223.0
2,BTG,B2gold Corp,Basic Materials,Gold,Canada,6.350000e+09,17.73,4.73,-0.0227,22847336.0
3,CITR,CitroTech Inc,Basic Materials,Specialty Chemicals,USA,1.738800e+08,NaN,9.08,0.0861,31118.0
4,CMCL,Caledonia Mining Corporation Plc,Basic Materials,Gold,United Kingdom,4.571400e+08,8.35,23.67,0.0059,112825.0


In [5]:
def company_data(ticker):
    stock= finvizfinance(ticker)
    stock_dict= stock.ticker_fundament()
    
    return stock_dict

stock_dict= company_data("AAPL")

In [6]:
def get_company_area(stock_dict):
    
    stock_areas= {"sector": stock_dict["Sector"],
                  "industry": stock_dict["Industry"],
                  "index":stock_dict["Index"].split(", ")
                  }
    
    return stock_areas

stock_areas= get_company_area(stock_dict)


In [19]:
def get_competition(stock_areas):
    
    # Get them and filter them
    fvaluation = Valuation()
    filters_dict = {'Sector': stock_areas["sector"]}
    fvaluation.set_filter(filters_dict=filters_dict)
    # Get DF
    all_competition_df = fvaluation.screener_view()
    
    fvaluation = Valuation()
    filters_dict = {'Sector': stock_areas["sector"], 'Industry': stock_areas["industry"]}
    fvaluation.set_filter(filters_dict=filters_dict)
    direct_competition_df= fvaluation.screener_view()
    
    indexes_dfs= get_index_companies(stock_areas)
    
    return {'all competition':all_competition_df,
            'direct competition': direct_competition_df, 
            'index competition':indexes_dfs
}
all_dfs = get_competition(stock_areas)

/opt/anaconda3/envs/quant_finance/lib/python3.12/site-packages/finvizfinance/screener/base.py:134: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  return pd.concat([df, pd.DataFrame(frame)], ignore_index=True)


/opt/anaconda3/envs/quant_finance/lib/python3.12/site-packages/finvizfinance/screener/base.py:134: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  return pd.concat([df, pd.DataFrame(frame)], ignore_index=True)


In [20]:
def get_index_companies(stock_areas):
    
    indexes= {}
    
    #hardcoded because they are only 4 opcions and dificult to get around it
    index_map = {
        "DJIA": "DJIA",
        "NDX": "NASDAQ 100",
        "S&P 500": "S&P 500",
        "RUT": "RUSSELL 2000"
        }
    
    for i in stock_areas["index"]:
        
        #converts the return of the index to find it in screnner
        mapped = index_map.get(i)
        if mapped is None:
            continue
        
        foverview = Overview()
        filters_dict = {'Index': mapped}
        foverview.set_filter(filters_dict=filters_dict)
        data = foverview.screener_view()
        
        indexes[mapped]= data
        
    return indexes
        

In [21]:
def filter_df(df, market_cap, past_sales_5, eps_next_5):
    df = df.copy()   
    if len(df) > 10:
        df = df[(df['Market Cap'] > market_cap/3) & (df['Market Cap'] < market_cap*3)]

    if len(df) > 10 and past_sales_5 != 0 and 'Sales Past 5Y' in df.columns:
        df['Sales Past 5Y'] = df['Sales Past 5Y'].str.replace('%', '').str.replace('-', '0').astype(float)
        df = df[abs(df['Sales Past 5Y'] - past_sales_5) < 10]

    if len(df) > 10 and eps_next_5 != 0 and 'EPS Next 5Y' in df.columns:
        df['EPS Next 5Y'] = df['EPS Next 5Y'].str.replace('%', '').str.replace('-', '0').astype(float)
        df = df[abs(df['EPS Next 5Y'] - eps_next_5) < 10]
        
    return df

In [22]:
def clean_dfs(all_dfs, ticker):
    
    # Values to evaluate

    df_all_competition= all_dfs['all competition']
    try: 
        market_cap = df_all_competition[df_all_competition['Ticker'] == ticker]['Market Cap'].iloc[0]
    except Exception:
        market_cap = 0
        
    if 'Sales Past 5Y' in df_all_competition.columns:
        try:
            past_sales_5 = df_all_competition[df_all_competition['Ticker'] == ticker]['Sales Past 5Y'].iloc[0]
            past_sales_5 = past_sales_5.replace("%", "").replace('-', "0")
            past_sales_5 = float(past_sales_5)
        except Exception:
            past_sales_5 = 0
    else:
        past_sales_5 = 0
        
    if 'EPS Next 5Y' in df_all_competition.columns:
        try:
            eps_next_5 = df_all_competition[df_all_competition['Ticker'] == ticker]['EPS Next 5Y'].iloc[0]
            eps_next_5 = eps_next_5.replace("%", "").replace('-', "0")
            eps_next_5 = float(eps_next_5)
        except Exception:
            eps_next_5 = 0
    else:
        eps_next_5 = 0
    
    df = filter_df(df_all_competition, market_cap, past_sales_5, eps_next_5)
        
    all_dfs['all competition']=  df
    
    #for the indexes
    df_indexes = all_dfs['index competition']
    
    for i in df_indexes:
        df= filter_df(df_indexes[i], market_cap, past_sales_5, eps_next_5)
        all_dfs[i]=  df
        
    all_dfs.pop("index competition")
    
    for i in all_dfs:
        df= all_dfs[i]["Ticker"]
        all_dfs[i]= df
                
    return all_dfs
        
    
stock_dict= company_data("AAPL")    
all_dfs_clean= clean_dfs(all_dfs, "AAPL")    


In [23]:
all_dfs_clean

{'all competition': 1      AAPL
 87     AVGO
 440    MSFT
 479    NVDA
 684     TSM
 Name: Ticker, dtype: object,
 'direct competition': 0     AAPL
 1     AXIL
 2     BOXL
 3     FEBO
 4     FOXX
 5     GMEX
 6     GPRO
 7     KOSS
 8      LPL
 9      MSN
 10    RIME
 11    SONO
 12    SONY
 13    TBCH
 14    UEIC
 15    VUZI
 16    WLDS
 17     WTO
 18    ZEPP
 Name: Ticker, dtype: object,
 'DJIA': 0     AAPL
 2     AMZN
 20    MSFT
 22    NVDA
 Name: Ticker, dtype: object,
 'NASDAQ 100': 0      AAPL
 11     AMZN
 15     AVGO
 43     GOOG
 44    GOOGL
 60     META
 64     MSFT
 68     NVDA
 90     TSLA
 Name: Ticker, dtype: object,
 'S&P 500': 1       AAPL
 31      AMZN
 45      AVGO
 212     GOOG
 213    GOOGL
 305     META
 320     MSFT
 340     NVDA
 447     TSLA
 Name: Ticker, dtype: object}

In [24]:
def all_values(all_dfs_clean, ticker):
   
    table_dfs = {}

    for key in all_dfs_clean: 
        table = []
        for t in all_dfs_clean[key]:  # t is each ticker string
            try:
                
                yfin = yf.Ticker(t).info
                market_cap = yfin.get('marketCap')
                total_debt = yfin.get('totalDebt', 0)
                total_cash = yfin.get('totalCash', 0)
                net_debt = total_debt - total_cash
                enterprise_value = yfin.get('enterpriseValue')
                minority_interest = enterprise_value - market_cap - net_debt
                total_revenue = yfin.get('totalRevenue')
                ebitda = yfin.get('ebitda')
                ev_revenue = enterprise_value / total_revenue
                ev_ebitda = enterprise_value / ebitda
                shares= yfin.get("sharesOutstanding")
                price=yfin.get("previousClose")
                
                temporal_dict = {
                    "ticker": t,  # peer ticker not subject
                    'market cap': market_cap,
                    'net debt': net_debt,
                    'minority interest': minority_interest,
                    'enterprise value': enterprise_value,
                    'total revenue': total_revenue,
                    'ebitda': ebitda,
                    'ev/revenue': ev_revenue,
                    'ev/ebitda': ev_ebitda, 
                    'shares':shares,
                    'price':price
  
            }
                
            except Exception:
                continue 
            table.append(temporal_dict)
        
        table_dfs[key] = pd.DataFrame(table)
    
    return table_dfs
    
    

In [25]:
a = all_values(all_dfs_clean, "AAPL")

In [26]:
a['all competition']

,ticker,market cap,net debt,minority interest,enterprise value,total revenue,ebitda,ev/revenue,ev/ebitda,shares,price
0,AAPL,3761492983808,23601999872,-4295618560,3780799365120,435617005568,152901992448,8.679182,24.726946,14681140000,255.63
1,AVGO,1491367493632,51882998784,-2077604864,1541172887552,68281999360,37218000896,22.570705,41.409341,4734668184,313.49
2,MSFT,2775695753216,33816002560,-2520203264,2806991552512,305453006848,175258992640,9.189602,16.016248,7425629076,369.37
3,NVDA,4311464017920,-51144000512,-887127040,4259432890368,215938007040,133230002176,19.725258,31.970523,24300000000,175.75
4,TSM,1758432591872,-2000078045184,7074929508352,6833284055040,3809054294016,2614264070144,1.793958,2.613846,5186504904,341.49


In [53]:
all_competition_df



,Ticker,Market Cap,P/E,Fwd P/E,PEG,P/S,P/B,P/C,P/FCF,EPS This Y,EPS Next Y,EPS Past 5Y,EPS Next 5Y,Sales Past 5Y,Price,Change,Volume
0,AAOI,8.570000e+09,NaN,24.55,NaN,18.80,11.64,39.65,NaN,462.31%,392.58%,24.98%,-,14.20%,113.90,0.1894,16895724.0
1,AAPL,3.694360e+12,31.84,27.05,2.41,8.48,41.95,55.22,29.96,13.66%,9.70%,17.91%,11.22%,8.71%,251.64,0.0006,45151784.0
2,ACFN,4.652000e+07,18.73,NaN,NaN,4.05,5.63,10.45,22.58,-,-,104.28%,-,14.15%,18.56,-0.0226,33214.0
3,ACIW,4.050000e+09,18.39,14.00,NaN,2.30,2.70,20.64,13.08,13.52%,16.15%,28.58%,-,6.34%,39.86,-0.0252,676922.0
4,ACLS,2.900000e+09,24.86,20.92,NaN,3.46,2.80,7.75,27.10,-25.24%,23.67%,20.99%,-,12.07%,94.39,0.1005,1300011.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
775,ZEPP,1.041200e+08,NaN,NaN,NaN,0.40,0.94,0.92,NaN,-,-,-,-,-22.56%,14.79,0.0602,139304.0
776,ZETA,3.940000e+09,NaN,13.60,0.47,3.02,4.91,12.33,21.30,29.40%,25.75%,17.79%,28.98%,28.80%,16.15,-0.0855,9959206.0
777,ZM,2.239000e+10,12.23,12.35,3.72,4.60,2.29,2.86,11.64,-0.74%,4.70%,22.38%,3.32%,12.92%,75.98,-0.0253,3778127.0
778,ZS,2.242000e+10,NaN,30.50,1.60,7.47,10.19,6.38,25.67,22.19%,14.03%,21.31%,19.03%,44.03%,139.41,-0.0816,3419263.0


In [55]:
direct_competition_df

,Ticker,Market Cap,P/E,Fwd P/E,PEG,P/S,P/B,P/C,P/FCF,EPS This Y,EPS Next Y,EPS Past 5Y,EPS Next 5Y,Sales Past 5Y,Price,Change,Volume
0,AAPL,3.694360e+12,31.84,27.05,2.41,8.48,41.95,55.22,29.96,13.66%,9.70%,17.91%,11.22%,8.71%,251.64,0.0006,45151784.0
1,AXIL,4.762000e+07,43.86,NaN,NaN,1.72,4.30,9.56,1190.50,-,-,-,-,91.76%,7.00,0.0638,55095.0
2,BOXL,9.800000e+05,NaN,NaN,NaN,0.01,NaN,0.08,NaN,-,-,15.55%,-,32.70%,1.03,0.0000,106182.0
3,FEBO,1.322000e+07,NaN,NaN,NaN,0.95,2.46,3.67,20.98,-,-,-,-,-,1.20,0.0214,1817.0
4,FOXX,2.949000e+07,NaN,NaN,NaN,0.47,NaN,16.66,NaN,-,-,-,-,-,4.21,-0.0047,4171.0
5,GMEX,1.450000e+06,NaN,NaN,NaN,0.43,0.03,0.77,NaN,-,-,-,-,-10.85%,1.03,-0.0804,2386298.0
6,GPRO,1.087000e+08,NaN,NaN,NaN,0.17,1.38,2.19,NaN,-,-,-5.64%,-,-6.09%,0.65,-0.0193,2599075.0
7,KOSS,3.758000e+07,NaN,NaN,NaN,2.94,1.24,2.43,NaN,-,-,-8.23%,-,-7.17%,3.97,-0.0100,19569.0
8,LPL,4.080000e+09,23.19,6.21,0.07,0.22,0.89,3.74,6.26,181.81%,46.31%,-,84.28%,-2.45%,4.08,-0.0490,2513366.0
9,MSN,7.870000e+06,NaN,NaN,NaN,1.19,0.51,0.59,NaN,-,-,-1.89%,-,11.38%,0.37,-0.0209,21542.0


In [39]:
indexes_dfs

{'DJIA':    Ticker                               Company                  Sector  \
 0    AAPL                             Apple Inc              Technology   
 1    AMGN                             AMGEN Inc              Healthcare   
 2    AMZN                        Amazon.com Inc       Consumer Cyclical   
 3     AXP                   American Express Co               Financial   
 4      BA                             Boeing Co             Industrials   
 5     CAT                       Caterpillar Inc             Industrials   
 6     CRM                        Salesforce Inc              Technology   
 7    CSCO                    Cisco Systems, Inc              Technology   
 8     CVX                          Chevron Corp                  Energy   
 9     DIS                        Walt Disney Co  Communication Services   
 10     GS              Goldman Sachs Group, Inc               Financial   
 11     HD                       Home Depot, Inc       Consumer Cyclical   
 12 

In [178]:
Yfin = yf.Ticker("AAPL").info
Yfin

{'address1': 'One Apple Park Way',
 'city': 'Cupertino',
 'state': 'CA',
 'zip': '95014',
 'country': 'United States',
 'phone': '(408) 996-1010',
 'website': 'https://www.apple.com',
 'industry': 'Consumer Electronics',
 'industryKey': 'consumer-electronics',
 'industryDisp': 'Consumer Electronics',
 'sector': 'Technology',
 'sectorKey': 'technology',
 'sectorDisp': 'Technology',
 'longBusinessSummary': 'Apple Inc. designs, manufactures, and markets smartphones, personal computers, tablets, wearables, and accessories worldwide. The company offers iPhone, a line of smartphones; Mac, a line of personal computers; iPad, a line of multi-purpose tablets; and wearables, home, and accessories comprising AirPods, Apple Vision Pro, Apple TV, Apple Watch, Beats products, and HomePod, as well as Apple branded and third-party accessories. It also provides AppleCare support and cloud services; and operates various platforms, including the App Store that allow customers to discover and download app

In [67]:
b= next(iter(a))
c= a[b].columns
for i in c:
    print(i)

ticker
market cap
net debt
minority interest
enterprise value
total revenue
ebitda
ev/revenue
ev/ebitda


In [27]:
all_dataframes= a.copy()
all_dataframes


{'all competition':   ticker     market cap       net debt  minority interest  enterprise value  \
 0   AAPL  3761492983808    23601999872        -4295618560     3780799365120   
 1   AVGO  1491367493632    51882998784        -2077604864     1541172887552   
 2   MSFT  2775695753216    33816002560        -2520203264     2806991552512   
 3   NVDA  4311464017920   -51144000512         -887127040     4259432890368   
 4    TSM  1758432591872 -2000078045184      7074929508352     6833284055040   
 
    total revenue         ebitda  ev/revenue  ev/ebitda       shares   price  
 0   435617005568   152901992448    8.679182  24.726946  14681140000  255.63  
 1    68281999360    37218000896   22.570705  41.409341   4734668184  313.49  
 2   305453006848   175258992640    9.189602  16.016248   7425629076  369.37  
 3   215938007040   133230002176   19.725258  31.970523  24300000000  175.75  
 4  3809054294016  2614264070144    1.793958   2.613846   5186504904  341.49  ,
 'direct competition':  

In [34]:
df= all_dataframes["direct competition"]
df = df.set_index("ticker")
df.head()



,market cap,net debt,minority interest,enterprise value,total revenue,ebitda,ev/revenue,ev/ebitda,shares,price
ticker,,,,,,,,,,
AAPL,3761492983808,23601999872,-4295618560,3780799365120,435617005568,152901992448,8.679182,24.726946,14681140000,255.63
AXIL,47891128,-4185734,2486,43707880,27664752,2174769,1.579912,20.097712,6802717,7.02
BOXL,1047061,32668000,28508999,62224060,106608000,-4255000,0.583672,-14.623751,951873,1.23
FEBO,12611250,-2264000,0,10347250,108715000,-14634000,0.095178,-0.707069,3062500,1.14
FOXX,39299128,19354548,-8,58653668,62266756,-8716156,0.941974,-6.729305,7005191,5.55


In [41]:
df= all_dataframes["direct competition"]
df = df.set_index("ticker")
df.head()

df = df.drop(columns=["shares", "price"])
stocks  = df.loc["AAPL"]
means   = df.mean(numeric_only=True)
medians = df.median(numeric_only=True)
expected  = df.loc["AAPL"].copy()
expected["enterprise value"]= stocks["ebitda"] * medians["ev/ebitda"]
expected["market cap"]= expected["enterprise value"] - stocks["net debt"] - stocks["minority interest"]
expected["ev/ebitda"]= medians["ev/ebitda"]
expected["ev/revenue"]= expected["enterprise value"] / expected["total revenue"]

comparison_df = pd.DataFrame({
    "Mean":     means,
    "Median":   medians,
    "AAPL":     stocks,
    "Expected": expected,
}).T

comparison_df.head()

,market cap,net debt,minority interest,enterprise value,total revenue,ebitda,ev/revenue,ev/ebitda
Mean,2.048910e+11,5.622476e+11,8.363000e+10,8.507686e+11,2.074716e+12,3.379840e+11,1.819003,-4.507050
Median,4.789113e+07,-2.264000e+06,0.000000e+00,5.865367e+07,1.087150e+08,-4.255000e+06,0.477575,0.234315
AAPL,3.761493e+12,2.360200e+10,-4.295619e+09,3.780799e+12,4.356170e+11,1.529020e+11,8.679182,24.726946
Expected,1.652092e+10,2.360200e+10,-4.295619e+09,3.582730e+10,4.356170e+11,1.529020e+11,0.082245,0.234315


In [ ]:
def streamlit_df(all_dataframes, ticker):
    
    streamlit_dfs={}
    for i in all_dataframes:
        df= all_dataframes[i]
        df = df.set_index("ticker")

        df = df.drop(columns=["shares", "price"])
        stocks  = df.loc[ticker]
        means   = df.mean(numeric_only=True)
        medians = df.median(numeric_only=True)
        expected  = df.loc[ticker].copy()
        expected["enterprise value"]= stocks["ebitda"] * medians["ev/ebitda"]
        expected["market cap"]= expected["enterprise value"] - stocks["net debt"] - stocks["minority interest"]
        expected["ev/ebitda"]= medians["ev/ebitda"]
        expected["ev/revenue"]= expected["enterprise value"] / expected["total revenue"]

        comparison_df = pd.DataFrame({
            "Mean":     means,
            "Median":   medians,
            ticker:     stocks,
            "Expected": expected,
        }).T
        streamlit_dfs[i]=comparison_df
    
    return streamlit_dfs

        

In [272]:
def build_table(all_dataframes, ticker):

    b= next(iter(all_dataframes))
    c= all_dataframes[b].columns
    
    for i in c:
        if i == "price" or i== "shares":
            pass
        else:
            print(f"{i:<20}|",end=" ")

    print("")
    print("_" *20* (len(c)-1))



    for dictionary in all_dataframes:
        print("")
        print(dictionary)
        
        df= all_dataframes[dictionary]
        row= df[df["ticker"] == ticker].iloc[0]
        
        df_no_ticker = df[df["ticker"] != ticker]
        
        for i in c: # Means
            if i == "ticker":
                print(f"{"Mean":<20}|",end=" ")
                
            elif i == "ev/revenue" or i == "ev/ebitda":
                mean= round(df_no_ticker[i].mean(),2)
                print(f"{mean:<20}|",end=" ")
            
            elif i== "price" or i== "shares":
                continue
                
            else:
                mean= round(df_no_ticker[i].mean()/1e6)
                print(f"{mean:<20}|",end=" ")
                
        print("")
        
        for i in c: #Median
            
            if i == "ticker":
                print(f"{"Median":<20}|",end=" ")
                
            elif i == "ev/revenue" or i == "ev/ebitda":
                median= round(df_no_ticker[i].median(),2)
                print(f"{median:<20}|",end=" ")
            
            elif i== "price" or i== "shares":
                continue
            
            else:
                median= round(df_no_ticker[i].median()/1e6)
                print(f"{(median):<20}|",end=" ")
                
        print("")
        
        for i in c: #Ticker
            if i == "ticker":
                print(f"{ticker:<20}|",end=" ")
                
            elif i == "ev/revenue" or i == "ev/ebitda":
                val= round(row[i],2)
                print(f"{val:<20}|",end=" ")
            
            elif i== "price" or i== "shares":
                continue
            
            else:
                val= round(row[i]/1e6)
                print(f"{(val):<20}|",end=" ")
                
        print("")
        print("- " *int(20/2)* (len(c)-1))
        
        df_no_ticker = df_no_ticker.copy()
        ev_ex= row["ebitda"]* df_no_ticker["ev/ebitda"].median()
        row["enterprise value"]= ev_ex
        row["market cap"]= ev_ex - row["net debt"] - row["minority interest"]
        row["ev/ebitda"]= df_no_ticker["ev/ebitda"].median()
                
        for i in c: #Expected
            
            if i == "ticker":
                print(f"{"Expected":<20}|",end=" ")
            
            elif i == "ev/revenue" or i == "ev/ebitda":
                val= round(row[i],2)
                print(f"{round(val,2):<20}|",end=" ")
                
            elif i== "price" or i== "shares":
                continue
            
            else:
                val= round(row[i]/1e6)
                print(f"{(round(val,2)):<20}|",end=" ")
                
        
        print("\n", "_" *20 *(len(c)-1))
    
        
        

In [108]:
all_dataframes= a 

In [296]:
def relative_valuation(ticker):
    
    stock_dict= company_data(ticker)
    print("Company data done")
    stock_areas= get_company_area(stock_dict)
    print("Company areas done")
    all_dfs= get_competition(stock_areas)
    print("Company competition done")
    all_dfs_clean= clean_dfs(all_dfs, ticker) 
    print("Company df clean done")   
    all_dataframes = all_values(all_dfs_clean, ticker)
    print("Company all values done")
    build_table(all_dataframes, ticker)
    
relative_valuation("NCLH")

Company data done
Company areas done
Company competition done##########################-] 25/26 
Company df clean done
Company all values done
ticker              | market cap          | net debt            | minority interest   | enterprise value    | total revenue       | ebitda              | ev/revenue          | ev/ebitda           | 
________________________________________________________________________________________________________________________________________________________________________________________________________

all competition
Mean                | 6945                | 476                 | 2000                | 9421                | 4867                | 809                 | 2.69                | 22.35               | 
Median              | 5245                | -534                | 13                  | 10530               | 4513                | 420                 | 2.33                | 24.04               | 
NCLH                | 9043                

In [278]:
Yfin = yf.Ticker("NVDA").info
Yfin
Yfin.get('totalRevenue')

215938007040

In [294]:
tickers = ["NOVT", "MSA", "NOV", "TEL", "DKS", "TSCO", "ULBI", "WINA", "NSSC", "GNTX"]

for ticker in tickers:
    try:
        print(ticker)
        relative_valuation(ticker)
    except Exception as ex:
        print(f"⚠️ {ticker} failed: {ex}")
        

NOVT
Company data done
Company areas done
Company competition done###########################] 96/97 
Company df clean done


/var/folders/hb/3snbzqyx6yd_96gxhv_h781w0000gn/T/ipykernel_71451/105685660.py:7: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['Sales Past 5Y'] = df['Sales Past 5Y'].str.replace('%', '').str.replace('-', '0').astype(float)


Company all values done
ticker              | market cap          | net debt            | minority interest   | enterprise value    | total revenue       | ebitda              | ev/revenue          | ev/ebitda           | 
________________________________________________________________________________________________________________________________________________________________________________________________________

all competition
Mean                | 4602                | 88313               | 11025               | 103940              | 206467              | 33840               | 4.1                 | 5.05                | 
Median              | 3563                | 68                  | 50                  | 4591                | 1223                | 260                 | 2.96                | 13.39               | 
NOVT                | 4228                | -78                 | -30                 | 4120                | 981                 | 178                 | 4.2    

/opt/anaconda3/envs/quant_finance/lib/python3.12/site-packages/finvizfinance/screener/base.py:134: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  return pd.concat([df, pd.DataFrame(frame)], ignore_index=True)
/var/folders/hb/3snbzqyx6yd_96gxhv_h781w0000gn/T/ipykernel_71451/105685660.py:7: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['Sales Past 5Y'] = df['Sales Past 5Y'].str.replace('%', '').str.replace('-', '0').astype(float)


Company competition done
Company df clean done
Company all values done
ticker              | market cap          | net debt            | minority interest   | enterprise value    | total revenue       | ebitda              | ev/revenue          | ev/ebitda           | 
________________________________________________________________________________________________________________________________________________________________________________________________________

all competition
Mean                | 7978                | 1990                | 865                 | 10833               | 5469                | 857                 | 66.19               | 12.78               | 
Median              | 6912                | 901                 | 155                 | 8865                | 3939                | 644                 | 2.35                | 13.69               | 
MSA                 | 6565                | 475                 | 264                 | 7304                | 1875

...

Company competition done###########################] 96/97 
Company df clean done


/var/folders/hb/3snbzqyx6yd_96gxhv_h781w0000gn/T/ipykernel_71451/105685660.py:7: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['Sales Past 5Y'] = df['Sales Past 5Y'].str.replace('%', '').str.replace('-', '0').astype(float)


Company all values done
ticker              | market cap          | net debt            | minority interest   | enterprise value    | total revenue       | ebitda              | ev/revenue          | ev/ebitda           | 
________________________________________________________________________________________________________________________________________________________________________________________________________

all competition
Mean                | 2020                | 599                 | 20                  | 2638                | 1476                | 251                 | 2.61                | 14.14               | 
Median              | 1878                | 306                 | 15                  | 2466                | 854                 | 176                 | 2.39                | 12.39               | 
NSSC                | 1388                | -110                | 92                  | 1370                | 192                 | 55                  | 7.13   

/var/folders/hb/3snbzqyx6yd_96gxhv_h781w0000gn/T/ipykernel_71451/105685660.py:7: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['Sales Past 5Y'] = df['Sales Past 5Y'].str.replace('%', '').str.replace('-', '0').astype(float)


Company all values done
ticker              | market cap          | net debt            | minority interest   | enterprise value    | total revenue       | ebitda              | ev/revenue          | ev/ebitda           | 
________________________________________________________________________________________________________________________________________________________________________________________________________

all competition
Mean                | 5359                | 2001                | 157                 | 7517                | 8656                | 970                 | 1.62                | 10.57               | 
Median              | 4411                | 1742                | 15                  | 6498                | 4976                | 746                 | 1.27                | 9.41                | 
GNTX                | 4828                | -139                | -101                | 4588                | 2534                | 590                 | 1.81   